# CourtLens 하이라이트 정제 모델 (Colab)

룰 베이스로 1차 가공된 KBL 하이라이트 클립에서 불필요한 구간을 잘라내는 모델을 학습하고 실행합니다.

- **학습**: NBA 공식 하이라이트(양성) vs 풀경기 랜덤 구간(음성) → 세그먼트 하이라이트 점수 분류기
- **추론**: KBL 1차 클립 → 초 단위 점수 → 임계값 컷 → 정제된 클립

런타임 유형을 **GPU(T4)** 로 설정하고 실행하세요.

In [ ]:
# 1. 코드 및 의존성 설치
!git clone -b ai-model https://github.com/gomja0124/likelion_CourtLens.git
%cd likelion_CourtLens/ai/highlight_refiner
!pip -q install -r requirements.txt

In [ ]:
# 2. CLIP 인코더 로드
import torch
from src.features import load_clip_encoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
clip_model, clip_preprocess = load_clip_encoder(device)

In [ ]:
# 3. 학습 데이터 수집 (URL을 팀에서 고른 영상으로 채우세요)
# 주의: 내부 연구/데모 용도로만 사용, 재배포 금지
from src.data_collect import download_videos

HIGHLIGHT_URLS = [
    # NBA 공식 하이라이트 (양성 예시) — 5~10개 권장
    # "https://www.youtube.com/watch?v=...",
]
FULLGAME_URLS = [
    # 풀경기/롱폼 중계 (음성 샘플용) — 2~3개면 충분
    # "https://www.youtube.com/watch?v=...",
]

highlight_files = download_videos(HIGHLIGHT_URLS, "data/highlights")
fullgame_files = download_videos(FULLGAME_URLS, "data/fullgames")
print(len(highlight_files), "highlight /", len(fullgame_files), "fullgame")

In [ ]:
# 4. 세그먼트 특징 추출 → 데이터셋 저장 (영상 양에 따라 10~30분)
from src.dataset import build_dataset

info = build_dataset(
    [str(p) for p in highlight_files],
    [str(p) for p in fullgame_files],
    clip_model, clip_preprocess,
    out_path="data/segments.npz",
    device=device,
)
info

In [ ]:
# 5. 학습 (수 분 소요, val AUC 0.85+ 목표)
from src.train import train_scorer

scorer = train_scorer("data/segments.npz", out_path="checkpoints/highlight_scorer.pt", device=device)

In [ ]:
# 6. KBL 1차 클립 업로드 → 정제 실행
from google.colab import files
from src.refine import refine_clip, plot_scores

uploaded = files.upload()  # 1차 가공된 KBL 클립(mp4) 업로드
kbl_clip = list(uploaded.keys())[0]

result = refine_clip(kbl_clip, scorer, clip_model, clip_preprocess, device=device)
print(f"{result['duration']}초 → {result['kept_duration']}초 (제거율 {result['removed_ratio']*100:.0f}%)")
print("keep 구간:", result["intervals"])
plot_scores(result)

In [ ]:
# 7. 결과 다운로드
files.download(result["output"])

## 임계값 튜닝

잘리는 게 너무 많으면 `hi`/`lo`를 낮추고, 쓸데없는 구간이 남으면 올리세요.

```python
result = refine_clip(kbl_clip, scorer, clip_model, clip_preprocess, device=device, hi=0.55, lo=0.4)
```

학습을 다시 하지 않아도 임계값만 바꿔 재실행할 수 있습니다.
체크포인트를 저장해두면 다음 세션에서 `src.train.load_scorer("checkpoints/highlight_scorer.pt")`로 바로 불러옵니다.